In [ ]:
import time

import random
import numpy as np

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error

import matplotlib.pyplot as plt

seed_model = 1.0 * 6742
seed_data_split = 1.0 * 1234
seed_data_split2 = 1.0 * 2749

In [ ]:
path = '' #Define the path to the data files here

## Data loading

In [ ]:
iinput = np.loadtxt('{}experimental_signals.txt'.format(path)) #Time signals usesd as input to the ML
input = np.loadtxt('{}experimental_signals.txt'.format(path))[30:] #Time signals usesd as input to the ML

ooutput_porosity = (1e-2 * np.loadtxt('{}porosity_experimental.txt'.format(path))).reshape(-1, 1)
output_porosity = (1e-2 * np.loadtxt('{}porosity_experimental.txt'.format(path)))[30:].reshape(-1, 1)

ooutput_wavespeed = np.loadtxt('{}wavespeed_experimental.txt').reshape(-1, 1)

for i in range(0, ooutput_wavespeed.shape[0], 30):
    ooutput_wavespeed[i:30 + i] = np.mean(ooutput_wavespeed[i:30 + i])

vmax = np.max(ooutput_wavespeed)
ooutput_wavespeed = ((vmax - ooutput_wavespeed) / vmax).reshape(-1, 1)
output_wavespeed = ooutput_wavespeed[30:].copy()

f = 5e6
wf = 2 * np.pi * f
k = wf / vmax

ooutput_pore_radius = np.loadtxt('{}poreRadius_experimental.txt'.format(path)).reshape(-1, 1)
output_pore_radius = np.loadtxt('{}poreRadius_experimental.txt'.format(path))[30:].reshape(-1, 1)
output_pore_radius = ((output_pore_radius ** 2) * output_porosity).reshape(-1, 1)

output_physics1 = np.zeros(output_porosity.shape)

## Physics constraints

In [ ]:
class Physics1(tf.keras.layers.Layer):
  def __init__(self):
    super(Physics1,self).__init__(name='eqn1')
    self.Anorm = self.add_weight(name="Anorm", shape=(), initializer=tf.keras.initializers.Constant(Anorm_init), trainable=True) # q
    self.Bnorm = self.add_weight(name="Bnorm", shape=(), initializer=tf.keras.initializers.Constant(Bnorm_init), trainable=True) # q
    
  def call(self, params):
    porosity, pore_radius, wavespeed = params
    
    term23 = tf.multiply(pore_radius, self.Bnorm)
    
    term24 = tf.multiply(porosity, self.Anorm)
    
    term25 = tf.add(term23, term24)
    
    eqn1 = wavespeed + term25

    return eqn1

## Feed-forward neural network definition

In [ ]:
def MLP():

  # Define inputs
  X_train = layers.Input(shape=(1303,),name='X_train')

  # Porosity + Pore radius model

  x = layers.Dense(256, kernel_initializer = 'normal', bias_initializer = 'zeros', activation="relu")(X_train)
  x = layers.Dense(128, kernel_initializer = 'normal', bias_initializer = 'zeros', activation="relu")(x)
  x = layers.Dense(64, kernel_initializer = 'normal', bias_initializer = 'zeros', activation="relu")(x) 
  x = layers.Dense(32, kernel_initializer = 'normal', bias_initializer = 'zeros', activation="relu")(x) 

  porosity = layers.Dense(1, kernel_initializer = 'glorot_normal', bias_initializer = 'zeros', activation="linear", name = "porosity")(x)

  pore_radius = layers.Dense(1, kernel_initializer = 'glorot_normal', bias_initializer = 'zeros', activation="linear", name = "pore_radius")(x)

  wavespeed = layers.Dense(1, kernel_initializer = 'glorot_normal', bias_initializer = 'zeros', activation="linear", name = "wavespeed")(x)
 
  eqn1 = Physics1()([porosity, pore_radius, wavespeed])
  
  model = keras.Model(inputs=[X_train],outputs=[porosity, pore_radius, wavespeed, eqn1])
  return model

## Data splitting into training validation and testing

In [ ]:
X_train, X_test, y_porosity_train, y_porosity_test,  y_pore_radius_train, y_pore_radius_test, y_wavespeed_train, y_wavespeed_test, y_physics1_train, y_physics1_test = train_test_split(input, output_porosity, output_pore_radius, output_wavespeed, output_physics1, test_size= 0.1, random_state = seed_data_split, shuffle=True)
X_train, X_val, y_porosity_train, y_porosity_val,  y_pore_radius_train, y_pore_radius_val, y_wavespeed_train, y_wavespeed_val, y_physics1_train, y_physics1_val = train_test_split(X_train, y_porosity_train, y_pore_radius_train, y_wavespeed_train, y_physics1_train, test_size= 0.11, random_state = seed_data_split, shuffle=True)

X_train_0, _, y_porosity_train_0, _,  y_pore_radius_train_0, _, y_wavespeed_train_0, _, y_physics1_train_0, _ = train_test_split(iinput[0:30], ooutput_porosity[0:30], ooutput_pore_radius[0:30], ooutput_wavespeed[0:30], output_physics1[0:30], test_size= 0.2, random_state = seed_data_split, shuffle=True)

if div < 1.0:
    _, X_train, _, y_porosity_train, _,  y_pore_radius_train, _, y_wavespeed_train, _, y_physics1_train = train_test_split(X_train, y_porosity_train, y_pore_radius_train, y_wavespeed_train, y_physics1_train, test_size= div, random_state = seed_data_split2, shuffle=True)

    _, X_train_0, _, y_porosity_train_0, _,  y_pore_radius_train_0, _, y_wavespeed_train_0, _, y_physics1_train_0 = train_test_split(X_train_0, y_porosity_train_0, y_pore_radius_train_0, y_wavespeed_train_0, y_physics1_train_0, test_size= div, random_state = seed_data_split2, shuffle=True)

X_train = np.vstack((X_train_0, X_train))
y_porosity_train = np.vstack((y_porosity_train_0, y_porosity_train))
y_pore_radius_train = np.vstack((y_pore_radius_train_0, y_pore_radius_train))
y_wavespeed_train = np.vstack((y_wavespeed_train_0, y_wavespeed_train))
y_physics1_train = np.vstack((y_physics1_train_0, y_physics1_train))

## Data normalization

In [ ]:
scaler_porosity = np.max(y_porosity_train)
y_porosity_train = y_porosity_train / scaler_porosity
y_porosity_val = y_porosity_val / scaler_porosity
y_porosity_test = y_porosity_test / scaler_porosity

scaler_pore_radius = np.max(y_porosity_pore_radius)
y_pore_radius_train = y_pore_radius_train / scaler_pore_radius
y_pore_radius_val = y_pore_radius_val / scaler_pore_radius
y_pore_radius_test = y_pore_radius_test / scaler_pore_radius

scaler_wavespeed = np.max(y_wavespeed_train)
y_wavespeed_train = y_wavespeed_train / scaler_wavespeed
y_wavespeed_val = y_wavespeed_val / scaler_wavespeed
y_wavespeed_test = y_wavespeed_test / scaler_wavespeed

## Model initialization

In [ ]:
learning_rate = 5e-4

model=MLP()

model.compile(loss={'porosity':'mse', 'pore_radius':'mse', 'wavespeed':'mse', 'eqn1':'mse'},
              optimizer=tf.keras.optimizers.Adam(learning_rate = learning_rate),
              metrics={'porosity':'mse', 'pore_radius':'mse', 'wavespeed':'mse', 'eqn1':'mse'},
              loss_weights=[1.0, 1.0, 1.0, 0.5])

## Model training

In [ ]:
training_epochs = 1000
batch_size = 5
patience = 7

callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights = True, start_from_epoch = 0)

weights_dict = {}
weight_callback = tf.keras.callbacks.LambdaCallback( on_epoch_end=lambda epoch, logs: weights_dict.update({epoch:model.get_weights()}))

time_start = time.time()

history=model.fit([X_train], [y_porosity_train, y_pore_radius_train, y_wavespeed_train, y_physics1_train], 
                  epochs=training_epochs, batch_size=batch_size, callbacks=[callback, weight_callback],
                  validation_data=([X_val], [y_porosity_val, y_pore_radius_val, y_wavespeed_val,  y_physics1_val]), verbose=1)  

time_taken = time.time() - time_start
print('\nTotal training time: {:.2f}s'.format(time_taken))

## Loss evolution over training epochs

In [ ]:
plt.figure(0, figsize=(9, 9))

plt.plot(history.history['loss'], linewidth = 4, label='Τraining')
plt.plot(history.history['val_loss'], linewidth = 4, label='Validation')

plt.xlabel('Epoch', fontsize = 40)
plt.ylabel('Loss', fontsize = 40)

plt.tick_params(axis='both', which='major', labelsize=40)
plt.legend(loc='upper right', fontsize = 30)
plt.show()

## r2-score & RMSE calculation

In [ ]:
def r2_score_RMSE_calc(prediction_type, scaler, X_train, y_train, X_val, y_val, X_test, y_test, model):
    
    if prediction_type == 'porosity':
        prediction_index = 0
    elif prediction_type == 'pore_radius':
        prediction_index = 1
    elif prediction_type == 'wavespeed':
        prediction_index == 2
    
    predictions_train = model.predict(X_train)[prediction_index]
    predictions_val = model.predict(X_val)[prediction_index]
    predictions_test = model.predict(X_test)[prediction_index]
    
    ppredictions_train = predictions_train * scaler
    ppredictions_val = predictions_val * scaler
    ppredictions_test = predictions_test * scaler
    
    yy_train = y_train * scaler
    yy_val = y_val * scaler
    yy_test = y_test * scaler
    
    r2_score_train = r2_score(yy_train, ppredictions_train)
    r2_score_val = r2_score(yy_val, ppredictions_val)
    r2_score_test = r2_score(yy_test, ppredictions_test)
    
    RMSE_train = np.sqrt(mean_squared_error(yy_train, ppredictions_train))
    RMSE_val = np.sqrt(mean_squared_error(yy_val, ppredictions_val))
    RMSE_test = np.sqrt(mean_squared_error(yy_test, ppredictions_test))
    
    print("r2-score scores: Train - %0.5f" %(r2_score_train))
    print("r2-score scores: Validation - %0.5f" %(r2_score_val))
    print("r2-score scores: Testing - %0.5f\n" %(r2_score_test))

    print("RMSE scores: Train - %0.5f" %(RMSE_train))
    print("RMSE scores: Validation - %0.5f" %(RMSE_val))
    print("RMSE scores: Testing - %0.5f\n" %(RMSE_test))
    
    fig, ax = plt.subplots(1, 3)
    fig.set_size_inches(30, 9)
    fig.tight_layout(pad = 5)
    
    ax[0].scatterplot(yy_train, ppredictions_train, s = 100, color = 'blue', edgecolor = 'black', edgewidth = 2)
    ax[0].plot(yy_train, np.interp(yy_train, yy_train, ppredictions_train), linewidth = 4, linestyle = 'dashed', color = 'black')
    
    ax[0].tick_params(axis='both', which='major', labelsize=20)
    ax[0].set_xlabel("True value", fontsize = 20, fontweight = 'bold')
    ax[0].set_ylabel("Predictions", fontsize = 20, fontweight = 'bold')
    ax[0].set_title("Training dataset", fontsize = 20, fontweight = 'bold')

    ax[1].scatterplot(yy_val, ppredictions_val, s = 100, color = 'blue', edgecolor = 'black', edgewidth = 2)
    ax[1].plot(yy_val, np.interp(yy_val, yy_val, ppredictions_val), linewidth = 4, linestyle = 'dashed', color = 'black')
    
    ax[1].tick_params(axis='both', which='major', labelsize=20)
    ax[1].set_xlabel("True value", fontsize = 20, fontweight = 'bold')
    ax[1].set_ylabel("Predictions", fontsize = 20, fontweight = 'bold')
    ax[1].set_title("Validation dataset", fontsize = 20, fontweight = 'bold')

    ax[2].scatterplot(yy_test, ppredictions_test, s = 100, color = 'blue', edgecolor = 'black', edgewidth = 2)
    ax[2].plot(yy_test, np.interp(yy_train, yy_test, ppredictions_test), linewidth = 4, linestyle = 'dashed', color = 'black')
    
    ax[2].tick_params(axis='both', which='major', labelsize=20)
    ax[2].set_xlabel("True value", fontsize = 20, fontweight = 'bold')
    ax[2].set_ylabel("Predictions", fontsize = 20, fontweight = 'bold')
    ax[2].set_title("Testing dataset", fontsize = 20, fontweight = 'bold')      
    
    return ;

## r2-score & RMSE porosity

In [ ]:
prediction_type = 'porosity'
scaler = scaler_porosity

r2_score_RMSE_calc(prediction_type, scaler, X_train, y_porosity_train, X_val, y_porosity_val, X_test, y_porosity_test, model)

## r2-score & RMSE squared pore radius

In [ ]:
prediction_type = 'pore_radius'
scaler = scaler_pore_radius

r2_score_RMSE_calc(prediction_type, scaler, X_train, y_pore_radius_train, X_val, y_pore_radius_val, X_test, y_pore_radius_test, model)

## r2-score & RMSE wave speed

In [ ]:
prediction_type = 'wavespeed'
scaler = scaler_wavespeed

r2_score_RMSE_calc(prediction_type, scaler, X_train, y_wavespeed_train, X_val, y_wavespeed_val, X_test, y_wavespeed_test, model)

## Calculation of true value for learnable parameters

In [ ]:
E_solid = 72.997166e9
v_solid = 0.335345
dens_solid = 2579.433958406887

cL_solid = np.sqrt(E_solid*(1-v_solid)/((1+v_solid)*(1-2*v_solid)*dens_solid))
cS_solid = np.sqrt(E_solid/(2 * (1+v_solid)*dens_solid))
wf=2*np.pi*f
kL = wf / cL_solid
kS = wf / cS_solid

q = kS / kL
A = (2 - (3 / 4) * q ** 2) + 5 / (1 - (9 / 4) * q ** 2)

B = - ((16 / 15) - (5 / 12) * q ** 2 + (3 / 16) * q ** 4) \
    + (7 / 15) / (1 - (19 / 12) * q ** 2) \
    - (5 / 9) * (5 - (9 / 4) * q ** 2) / ((1 -(9 / 4) * q ** 2) ** 2)

Anorm_true = A * 0.5 * scaler_porosity / scaler_wavespeed
Bnorm_true = B * 0.5 * (k ** 2) * scaler_pore_radius / scaler_wavespeed

## Learnable parameters evolution over training epochs

In [ ]:
names = [weight.name for layer in model.layers for weight in layer.weights]
weights = model.get_weights()

for i in range(len(names)):
    if names[i][0:len("Anorm")] == "Anorm":
        Anorm_pred = model.get_weights()[i]
    elif names[i][0:len("Bnorm")] == "Bnorm":
        index_Bnorm = i
        
history_Anorm = []
history_Bnorm = []

for epoch, weights in weights_dict.items():
    epoch_num.append(epoch + 1)
    history_Anorm.append(weights[index_Anorm])
    history_Bnorm.append(weights[index_Bnorm])
    
print("The predicted and true values for A_norm are %f and %f respectively" %(history_Anorm[-1], Anorm_true))
print("The predicted and true values for B_norm are %f and %f respectively" %(history_Bnorm[-1], Bnorm_true))
    
fig, ax = plt.subplots(1, 2)
fig.set_size_inches(20, 9)
fig.tight_layout(pad = 5)

ax[0].plot([0, epoch_num[-1]], [Anorm_true, Anorm_true], '--', color = 'black', label = 'A_norm,true')
ax[0].plot(epoch_num, history_Anorm, color = 'black', label = 'A_norm')
ax[0].tick_params(axis='both', which='major', labelsize=20)
ax[0].set_xlabel("Epoch", fontsize = 20, fontweight = 'bold')
ax[0].set_ylabel("Predictions", fontsize = 20, fontweight = 'bold')
ax[0].legend(fontsize = 20)

ax[1].plot([0, epoch_num[-1]], [Bnorm_true, Bnorm_true], '--', color = 'blue', label = 'B_norm,true')
ax[1].plot(epoch_num, history_Bnorm, color = 'blue', label = 'B_norm')
ax[1].tick_params(axis='both', which='major', labelsize=20)
ax[1].set_xlabel("Epoch", fontsize = 20, fontweight = 'bold')
ax[1].set_ylabel("Predictions", fontsize = 20, fontweight = 'bold')
ax[1].legend(fontsize = 20)